# Tic-Tac-Toe Reinforcement Learning

This notebook builds a Tic-Tac-Toe game and teaches an agent how to play.

In this notebook, I:
- created the game board and the rules;
- added a random player for testing;
- created a reinforcement learning agent that uses a Q-table;
- trained the agent by playing many games;
- tested the learned agent against a human player.


## Creating the game environment

I created the game board and added functions to display it, reset it, find empty spaces, and place a move. The code also checks for a winner or a draw. Finally, it gives the agent a reward: 1 for a win, -1 for a loss, and 0 for a draw or unfinished game.

In [1]:
import random
class Environment:
    # Purpose of this method is to iniitialize a board
    def __init__(self):
        self.board = [
            ["X", "O", "O"],
            ["", "", ""],
            ["", "O", "X"]
        ]
        self.action_space = [
           (row, col)
           for row in range(3)
           for col in range(3)
        ]
        self.rl_agent_player = "X"

    # Purpose of this method is to print the board
    def render(self):
        for i in range(0,3):
            for j in range(0,3):
                if self.board[i][j] == "":
                    print(" ", end = "|")
                else:
                    print(self.board[i][j], end = "|")
            print("")

    # Purpose of this method is to get the current state of the board
    def get_state(self):
        return [row[:] for row in self.board];

    # Purpose of this method is to see the available actions
    def get_available_actions(self):
        list_of_possible_action = []
        for i in range(0, 3):
            for j in range(0,3):
                if self.board[i][j] == "":
                    list_of_possible_action.append((i,j))
        return list_of_possible_action                    

    # Purpose of this method is to reset the environment
    def reset(self):
        self.board = [
            ["", "", ""],
            ["", "", ""],
            ["", "", ""]
        ]
        return self.get_state()

    # Purpose of this method is to execute the action received
    def step(self, action_pos, action_val):
        if action_pos not in self.get_available_actions():
            raise ValueError("Invalid action")
        (row, col) = action_pos
        self.board[row][col] = action_val
        return self.board

    # Purpose of this function is to define the one episode of tic tac toe game is over or not
    def is_done(self):
        result = (
            self.is_done_columns(),
            self.is_done_rows(),
            self.is_done_diagonal()
        )
        
        for game_over, winner in result:
            if game_over:
                return (True, winner)

        return self.is_draw()
    
    # Helper Method
    def is_done_columns(self):
        for col in range(0,3):
            if self.board[0][col] != "" and self.board[0][col] == self.board[1][col] and self.board[1][col] == self.board[2][col]:
                return (True, self.board[0][col]);
        return (False, None);

    # Helper Method
    def is_done_rows(self):
         for row in range(0,3):
            if self.board[row][0] != "" and self.board[row][0] == self.board[row][1] and self.board[row][1] == self.board[row][2]:
                return (True, self.board[row][0]);
         return (False, None);

    # Helper Method
    def is_done_diagonal(self):
        if self.board[0][0] != "" and self.board[0][0] == self.board[1][1] and self.board[1][1] == self.board[2][2]:
            return (True, self.board[1][1])
        if self.board[0][2] != "" and self.board[0][2] == self.board[1][1] and self.board[1][1] == self.board[2][0]:
            return (True, self.board[1][1])
        return (False, None);

    # Helper Method
    def is_draw(self):
        action_list = self.get_available_actions()
        if len(action_list) == 0:
            return (True, None)
        return (False, None)

    # Purpose of this method is to define the reward function
    def get_reward(self):
        check_is_done = self.is_done()
        if check_is_done[0] == False:
            return 0;
        else:
            if check_is_done[1] == None:
                return 0;
            elif check_is_done[1] == self.rl_agent_player:
                return 1;
            else:
                return -1;

## Adding a random player

I added a simple player that chooses an empty square at random. This player helps test that the game rules and turns work correctly.

In [2]:
class RandomAgent:
    def __init__(self):
        pass
        
    def select_action(self, env):
        possible_actions = env.get_available_actions();
        return random.choice(possible_actions)

    def step(self, env, action_pos):
        env.step(action_pos, "O")

In [3]:
def play_random_game():
    env = Environment()
    agent = RandomAgent()

    env.reset()
    current_player = "X"

    print("Starting Tic-Tac-Toe Game")
    env.render()

    while True:
        print(f"\nPlayer {current_player}'s turn")

        # Select a random available action
        action = agent.select_action(env)

        # Execute the action
        env.step(action, current_player)

        # Display updated board
        env.render()

        # Check whether the game has ended
        game_over, winner = env.is_done()

        if game_over:
            if winner is None:
                print("\nResult: DRAW!")
            else:
                print(f"\nWinner: Player {winner}")
            break

        # Switch player
        current_player = "O" if current_player == "X" else "X"


# Run 10 games
for game in range(2):
    print(f"\n{'=' * 30}")
    print(f"GAME {game + 1}")
    print(f"{'=' * 30}")

    play_random_game()



GAME 1
Starting Tic-Tac-Toe Game
 | | |
 | | |
 | | |

Player X's turn
 | | |
 | | |
 | |X|

Player O's turn
 | | |
 | |O|
 | |X|

Player X's turn
 | | |
 | |O|
 |X|X|

Player O's turn
 | |O|
 | |O|
 |X|X|

Player X's turn
X| |O|
 | |O|
 |X|X|

Player O's turn
X| |O|
O| |O|
 |X|X|

Player X's turn
X| |O|
O|X|O|
 |X|X|

Winner: Player X

GAME 2
Starting Tic-Tac-Toe Game
 | | |
 | | |
 | | |

Player X's turn
 | | |
 | | |
X| | |

Player O's turn
 | |O|
 | | |
X| | |

Player X's turn
 |X|O|
 | | |
X| | |

Player O's turn
 |X|O|
 | |O|
X| | |

Player X's turn
 |X|O|
X| |O|
X| | |

Player O's turn
 |X|O|
X| |O|
X|O| |

Player X's turn
 |X|O|
X|X|O|
X|O| |

Player O's turn
O|X|O|
X|X|O|
X|O| |

Player X's turn
O|X|O|
X|X|O|
X|O|X|

Result: DRAW!


## Testing random games

I ran several games between random players. This checks that the board changes, turns alternate, and the game stops when there is a winner or a draw.

## Creating the learning agent

I created an agent that stores values for different board states in a Q-table. It sometimes tries a random move and sometimes chooses the move with the best value. After each move, it updates the table using the result of the game.

In [4]:
class RLAgent:
    def __init__(self):
        self.q_table = {}
        self.learning_rate = 0.1
        self.discount_factor = 0.9
        self.epsilon = 0.1
    
    def get_state_key(self, state):
     key = ""
     for row in range(3):
         for col in range(3):
             if state[row][col] == "":
                key += "-"
             else:
                key += state[row][col]
     return key;

    # Initialise Q-values for available actions
    def initialise_state(self, state, available_actions):
        key = self.get_state_key(state)
        if key not in self.q_table:
            value_dict = {}
            for action in available_actions:
                value_dict[action] = 0
            self.q_table[key] = value_dict

    def select_action(self, state, available_actions):
        self.initialise_state(state, available_actions)
        random_num = random.random()

        # Exploration
        if random_num < self.epsilon:
            action = random.choice(available_actions)

        # Exploitation
        else:
            state_key = self.get_state_key(state)
            q_values = self.q_table[state_key]
            max_q_value = max(q_values.values())
            best_actions = [
                action
                for action, value in q_values.items()
                if value == max_q_value
            ]
            action = random.choice(best_actions)
        return action

    def update_q_value(self, state, action, reward, next_state, next_available_actions, done):
        current_state_key = self.get_state_key(state)
        current_q_value = self.q_table[current_state_key][action]
        
        # The game is over, in this case we dont need the future value
        if done:
            target = reward
        else:
            next_state_key = self.get_state_key(next_state)
            # Initialise next state if it hasn't been visited
            self.initialise_state(next_state, next_available_actions)
            max_q_value = max(self.q_table[next_state_key].values())
            target = reward + self.discount_factor * max_q_value
        
        # Update Q-value
        self.q_table[current_state_key][action] = (current_q_value + self.learning_rate * (target - current_q_value))

## Training the agent

I trained the agent by making it play 100,000 games against the random player. The agent learns from wins, losses, and draws. The code also records the results so I can see how well it learned.

In [5]:
def train_agent(total_episodes):
    env = Environment()
    agent = RLAgent()
    opponent = RandomAgent()

    # Training statistics
    wins = 0
    losses = 0
    draws = 0

    for episode in range(total_episodes):
        env.reset()
        done = False
        winner = None

        while not done:
            # 1. RL Agent's turn (X)
            old_state = env.get_state()

            selected_action = agent.select_action(old_state, env.get_available_actions())

            env.step(selected_action, "X")
            done, winner = env.is_done()

            # Check if RL agent has won
            if done:
                reward = env.get_reward()

                agent.update_q_value(old_state, selected_action, reward, env.get_state(), [], True)
                break

            # 2. Random Opponent's turn (O)
            opponent_action = opponent.select_action(env)
            env.step(opponent_action, "O")
            done, winner = env.is_done()

           
            # 3. Update Q-value
            reward = env.get_reward()
            next_state = env.get_state()
            next_available_actions = env.get_available_actions()
            agent.update_q_value(old_state, selected_action, reward, next_state, next_available_actions, done)

        # 4. Record episode result
        if winner == "X":
            wins += 1
        elif winner == "O":
            losses += 1
        else:
            draws += 1

    print("Training completed!")
    print(f"Total episodes: {total_episodes}")
    print(f"Wins: {wins}")
    print(f"Losses: {losses}")
    print(f"Draws: {draws}")

    print(f"Win rate: {wins / total_episodes * 100:.2f}%")
    print(f"Draw rate: {draws / total_episodes * 100:.2f}%")

    return agent


In [6]:
agent = train_agent(100000)

Training completed!
Total episodes: 100000
Wins: 92289
Losses: 3144
Draws: 4567
Win rate: 92.29%
Draw rate: 4.57%


In [7]:
print("Number of learned states:", len(agent.q_table))

Number of learned states: 2151


In [8]:
def play_against_agent(agent):
    env = Environment()
    env.reset()

    # Disable exploration
    original_epsilon = agent.epsilon
    agent.epsilon = 0

    done = False
    winner = None

    print("You are O. RL Agent is X.")
    print("Enter row and column between 0 and 2.")

    while not done:

        # RL Agent's turn
        print("\nRL Agent's turn:")

        state = env.get_state()
        available_actions = env.get_available_actions()

        selected_action = agent.select_action(state, available_actions)

        print("Agent selected:", selected_action)

        env.step(selected_action, "X")
        env.render()

        done, winner = env.is_done()

        if done:
            break

        # Your turn
        print("\nYour turn (O)")

        available_actions = env.get_available_actions()

        while True:
            try:
                row = int(input("Enter row (0-2): "))
                col = int(input("Enter column (0-2): "))

                if (row, col) in available_actions:
                    break

                print("Invalid position! Choose an empty cell.")

            except ValueError:
                print("Please enter valid numbers.")

        env.step((row, col), "O")
        env.render()

        done, winner = env.is_done()

    # Display result
    print("\n========== GAME OVER ==========")

    if winner == "X":
        print("RL Agent wins!")
    elif winner == "O":
        print("Congratulations! You win!")
    else:
        print("It's a draw!")

    agent.epsilon = original_epsilon


## Playing against the trained agent

I added a final game where a human can play against the trained agent. This shows whether the agent can use what it learned to choose moves in a new game.

In [9]:
play_against_agent(agent)

You are O. RL Agent is X.
Enter row and column between 0 and 2.

RL Agent's turn:
Agent selected: (0, 1)
 |X| |
 | | |
 | | |

Your turn (O)


Enter row (0-2):  0
Enter column (0-2):  0


O|X| |
 | | |
 | | |

RL Agent's turn:
Agent selected: (1, 1)
O|X| |
 |X| |
 | | |

Your turn (O)


Enter row (0-2):  2
Enter column (0-2):  2


O|X| |
 |X| |
 | |O|

RL Agent's turn:
Agent selected: (2, 1)
O|X| |
 |X| |
 |X|O|

========== GAME OVER ==========
RL Agent wins!


Enter row (0-2):  1
Enter column (0-2):  1


... (3 outputs left out)

Enter row (0-2):  2
Enter column (0-2):  1


 | |O|
 |O|X|
X|O|X|

RL Agent's turn:
Agent selected: (0, 1)
 |X|O|
 |O|X|
X|O|X|

Your turn (O)


Enter row (0-2):  0
Enter column (0-2):  0


O|X|O|
 |O|X|
X|O|X|

RL Agent's turn:
Agent selected: (1, 0)
O|X|O|
X|O|X|
X|O|X|

========== GAME OVER ==========
It's a draw!
